In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;

Init();

In [ ]:
string proj = "TemperatureVelocityCoupling";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();


In [ ]:
// int[] kelemSeq = new[] {16};//new int[]{4, 8, 16, 32, 64, 128, 256};
int[] kelemSeq = new int[]{4, 8, 16, 32, 64};
var GridSeq = new IGridInfo[kelemSeq.Length];

In [ ]:
for(int iGrid = 0; iGrid < GridSeq.Length; iGrid++) {
    
    int kelem = kelemSeq[iGrid];
    
    GridCommons grd;   
    double[] xNodes = GenericBlas.Linspace(-3.0/2.0, 3.0/2.0, 2 * kelem + 1);
    double[] yNodes = GenericBlas.Linspace(-3.0/2.0, 3.0/2.0, 2 * kelem + 1);    
    grd = Grid2D.Cartesian2DGrid(xNodes, yNodes);

    // HIER ANPASSUNG DER RANDBEDINGUNGEN!
    grd.EdgeTagNames.Add(1, "NavierSlip_linear_ConstantTemperature_upper");
    grd.EdgeTagNames.Add(2, "NavierSlip_linear_ConstantTemperature_lower");
    grd.EdgeTagNames.Add(3, "NavierSlip_linear_ConstantTemperature_left");
    grd.EdgeTagNames.Add(4, "NavierSlip_linear_ConstantTemperature_right");
 
    grd.DefineEdgeTags(delegate (double[] X) {
        byte et = 0;
        if (Math.Abs(X[1] - yNodes.Last()) <= 1.0e-8) // UPPER BOUNDARY
            et = 1;
        if (Math.Abs(X[1] - yNodes.First()) <= 1.0e-8) // LOWER BOUNDARY
            et = 2;
        if (Math.Abs(X[0] - xNodes.First()) <= 1.0e-8) // LEFT BOUNDARY
            et = 3;
        if (Math.Abs(X[0] - xNodes.Last()) <= 1.0e-8) // RIGHT BOUNDARY
            et = 4;
        return et;
    });

    // grd.Name = "StaticDropletOnWall_meshStudy";
    grd.Name = "TemperatureVelocityCoupling_" + 2*kelem;        
    
    BoSSSshell.WorkflowMgm.DefaultDatabase.SaveGrid(ref grd);
    
    GridSeq[iGrid] = grd;
}

In [ ]:
public static List<XNSFE_Control> TemperatureVelocityCoupling(int[] pDegS, IGridInfo[] grdS, int setup){
    
    List<XNSFE_Control> controls = new List<XNSFE_Control>();

    foreach(int pDeg in pDegS){
        for(int tDeg = pDeg-1; tDeg <= pDeg + 2; tDeg++){
            foreach(IGridInfo grd in grdS){
                controls.Add(TemperatureVelocityCoupling(pDeg, tDeg, grd, setup));
            }
        }
    }
    
    return controls;    
}

public static XNSFE_Control TemperatureVelocityCoupling(int pDeg, int tDeg, IGridInfo grd, int setup) {
    
    XNSFE_Control C = new XNSFE_Control();
    
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("case", setup));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("degreeU", pDeg));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("degreeT", tDeg));
    C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("gridlevel", ((GridCommons)grd).Name.Split('_').Last()));
    
    C.savetodb = true;
    C.ContinueOnIoError = false;

    // misc. solver options
    // ====================
    C.NonLinearSolver.MaxSolverIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-6;

    // Level-Set options (AMR)
    // =======================
    C.LSContiProjectionMethod = BoSSS.Solution.LevelSetTools.ContinuityProjectionOption.None;
    C.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.None;
    C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;


    Dictionary<byte, string> Boundaries = new Dictionary<byte, string>();
    Boundaries = new Dictionary<byte, string>() {
        { 1, "NavierSlip_linear_ConstantTemperature_upper"},
        { 2, "NavierSlip_linear_ConstantTemperature_lower"},
        { 3, "NavierSlip_linear_ConstantTemperature_left"},
        { 4, "NavierSlip_linear_ConstantTemperature_right"},
    };

    Formula TFunc;
    switch(setup){
        case 0:
            TFunc = new Formula("X => X[0]");
            break;
        case 1:
            TFunc = new Formula("X => X[0]*X[1]");
            break;
        case 2:
        default:
            TFunc = new Formula("X => Math.Sin(Math.PI * X[0] / 3.0) * Math.Sin(Math.PI * X[1] / 3.0)");
            break;
    }
    
    C.AddBoundaryValue(Boundaries[1], "Temperature#B", TFunc);
    C.AddBoundaryValue(Boundaries[2], "Temperature#B", TFunc);
    C.AddBoundaryValue(Boundaries[3], "Temperature#B", TFunc);
    C.AddBoundaryValue(Boundaries[4], "Temperature#B", TFunc);

    C.SetDGdegree(pDeg);
    C.FieldOptions[BoSSS.Solution.NSECommon.VariableNames.Temperature].Degree = tDeg;
    C.SetGrid(grd);

    // Physical Parameters
    // ===================
    C.PhysicalParameters.IncludeConvection = false;
    C.PhysicalParameters.Material = false;
    C.PhysicalParameters.rho_A = 1.0;
    C.PhysicalParameters.rho_B = 0.1; // unequal density
    C.PhysicalParameters.mu_A = 0.5;
    C.PhysicalParameters.mu_B = 0.05;
    C.PhysicalParameters.Sigma = 0.0;
    C.PhysicalParameters.theta_e = Math.PI / 2.0;

    C.PhysicalParameters.betaS_A = 0.0;
    C.PhysicalParameters.betaS_B = 0.0;
    C.PhysicalParameters.betaL = 0.0;

    C.ThermalParameters.IncludeConvection = false;
    C.IncludeRecoilPressure = false;
    C.ThermalParameters.rho_A = C.PhysicalParameters.rho_A;
    C.ThermalParameters.rho_B = C.PhysicalParameters.rho_B;
    C.ThermalParameters.k_A = 1.0;
    C.ThermalParameters.k_B = 0.1;
    C.ThermalParameters.hVap = 1000.0; // 0.0 means no evaporation -  for reference case!
    C.ThermalParameters.T_sat = 0.0;

    C.ThermalParameters.sliplength = 0.0;
    C.PhysicalParameters.slipI = 0.0;

    C.LinearSolver = LinearSolverCode.direct_pardiso.GetConfig();
    C.NoOfMultigridLevels = 1;
    if (C.LinearSolver is OrthoMGSchwarzConfig) {
        ((OrthoMGSchwarzConfig)C.LinearSolver).CoarseKickIn = 90000;
        ((OrthoMGSchwarzConfig)C.LinearSolver).TargetBlockSize = 10000;
        ((OrthoMGSchwarzConfig)C.LinearSolver).CoarseUsepTG = false;
        ((OrthoMGSchwarzConfig)C.LinearSolver).UsepTG = false;
        ((OrthoMGSchwarzConfig)C.LinearSolver).ConvergenceCriterion = 1e-12;
    }

    double r = 0.8;
    C.AddInitialValue("Phi", $"X => (X[0]).Pow2() /({r}).Pow2() + (X[1]).Pow2() / ({r}).Pow2() - 1.0", false);

    // MAKE TIMESTEPPING SETTINGS MORE CLEAR!
    // Timestepping
    // ============
    C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
    C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
    C.Timestepper_BDFinit = BoSSS.Solution.Timestepping.TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = BoSSS.Solution.XdgTimestepping.LevelSetHandling.None;

    C.SessionName = ((GridCommons)grd).Name + //grid
        "_ku" + pDeg +// degree u
        "_kT" + tDeg + // degree T
        "_Case" + setup; // B.C. 

    //C.TracingNamespaces = "*";

    return C;
}

In [ ]:
int[] degS = new int[] { 2, 3, 4};

In [ ]:
int iDeg0 = 0;
int iGrd0 = 0;

In [ ]:
List<XNSFE_Control> controls = new List<XNSFE_Control>();

In [ ]:
// Testcase setup
// ==============
{
    var ctrls = TemperatureVelocityCoupling(degS,GridSeq,0);
    controls.AddRange(ctrls);
}
{
    var ctrls = TemperatureVelocityCoupling(degS,GridSeq,1);
    controls.AddRange(ctrls);
}
{
    var ctrls = TemperatureVelocityCoupling(degS,GridSeq,2);
    controls.AddRange(ctrls);
}

Console.WriteLine("# of controls: " + controls.Count());
int ctrlCnt = controls.Count();

In [ ]:
// // FOR TESTING IF THIS RUNS
// foreach(var C in controls){
// var C = controls.ElementAt(80);
// C.ImmediatePlotPeriod = 1;
// C.SuperSampling = 2;
// C.savetodb = false;
// C.PostprocessingModules.Clear();
// using(var solver = new XNSFE()){
//     solver.Init(C);
//     solver.RunSolverMode();
// }
// }


In [ ]:
bool run      = true;
var bpc = BoSSSshell.GetDefaultQueue();

In [ ]:
//BoSSSshell.WorkflowMgm.ResetProject(true, true, true, true);

In [ ]:
if(BoSSSshell.WorkflowMgm.AllJobs.Count() > 0){
     BoSSSshell.WorkflowMgm.ResetProject();
}
var jobs = controls.Select(c => c.CreateJob()).ToArray();
jobs.ForEach(j => j.NumberOfThreads = 1);
jobs.ForEach(j => j.EnvironmentVars["BOSSS_ARG_7"] = j.NumberOfThreads.ToString());

In [ ]:
jobs.Activate(bpc);

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(7200);

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions.ForEach(s => display(s));

In [ ]:
sessions = sessions.Where(s => s.SuccessfulTermination).ToArray();
sessions.Count()

In [ ]:
int count = BoSSSshell.wmg.Sessions.Count();
int success = BoSSSshell.wmg.Sessions.Where(s => s.SuccessfulTermination).Count();

if(count != ctrlCnt || count != success){
    throw new ApplicationException("Not all simulations calculated or finished successful!");
}

In [ ]:
// Für Vislt notwendig

//sessions.ForEach(session=> session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).Do());
// session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).WithShadowFields().Do();
// session.Export().To(Path.GetFullPath(Path.Join(".",session.Name))).WithSupersampling(2).Do();

In [ ]:
public static Dictionary<string, Plot2Ddata> ExtractConvergenceH(IEnumerable<ISessionInfo> filteredsessions){

    Dictionary<string, Plot2Ddata> ret = new Dictionary<string, Plot2Ddata>();
    
    // ITimestepInfo[] timesteps = filteredsessions.Select(s => s.Timesteps.Last()).ToArray();
    // string[] fields = timesteps.First().Fields.Select(f => f.Identification).ToArray();
    string[] fields = new string[] {"Pressure", "VelocityX", "VelocityY", "Temperature"};
    
    foreach(var field in fields){
        Plot2Ddata errDat = filteredsessions.ToEstimatedGridConvergenceData(field, normType: NormType.L2_embedded); 
        ret.Add(field, errDat);
    } 
    
    return ret;
}

public static Dictionary<string, List<ISessionInfo>> ExtractSessionsByName(IEnumerable<ISessionInfo> sessions){
    
    Dictionary<string, List<ISessionInfo>> filteredsessions = new Dictionary<string, List<ISessionInfo>>();
    
    foreach(var sess in sessions){
        var key = Regex.Split(sess.Name , ".*TemperatureVelocityCoupling_[0-9]+_",RegexOptions.IgnoreCase).Last();
        
        if(!filteredsessions.ContainsKey(key)){
            filteredsessions[key] = new List<ISessionInfo>();
        }
        filteredsessions[key].Add(sess);
    }
    
    return filteredsessions;
}

In [ ]:
using System.Data;

In [ ]:
DataTable table = new DataTable("ConvergenceData");
DataColumn column;
DataRow row;

column = new DataColumn();
column.DataType = System.Type.GetType("System.String");
column.ColumnName = "Study";
column.ReadOnly = true;
column.Unique = true;
table.Columns.Add(column);

column = new DataColumn();
column.DataType = typeof(List<ISessionInfo>);
column.ColumnName = "Sessions";
column.ReadOnly = false;
column.Unique = false;
table.Columns.Add(column);

table.Columns.Add("Timesteps", typeof(List<ITimestepInfo>));
table.Columns.Add("SessionID", typeof(List<Guid>));

table.Columns.Add("P-Convergence", typeof(Dictionary<int, double>));
table.Columns.Add("P-PlotData", typeof(Plot2Ddata));

table.Columns.Add("U-Convergence", typeof(Dictionary<int, double>));
table.Columns.Add("U-PlotData", typeof(Plot2Ddata));

table.Columns.Add("V-Convergence", typeof(Dictionary<int, double>));
table.Columns.Add("V-PlotData", typeof(Plot2Ddata));

table.Columns.Add("Velocity-Convergence", typeof(Dictionary<int, double>));
table.Columns.Add("Velocity-PlotData", typeof(Plot2Ddata));

table.Columns.Add("T-Convergence", typeof(Dictionary<int, double>));
table.Columns.Add("T-PlotData", typeof(Plot2Ddata));

var filteredsessions = ExtractSessionsByName(sessions);

foreach(var kvp in filteredsessions){
    row = table.NewRow();
    row["Study"] = kvp.Key;
    row["Sessions"] = kvp.Value;
    table.Rows.Add(row);
}

In [ ]:
foreach(DataRow row in table.Rows){
    List<ITimestepInfo> ts = new List<ITimestepInfo>();
    List<Guid> ids = new List<Guid>();

    foreach(ISessionInfo s in (List<ISessionInfo>)row["Sessions"]){
        ts.Add(s.Timesteps.Last());
        ids.Add(s.ID);
    }

    row["Timesteps"] = ts;
    row["SessionID"] = ids;

    row["P-PlotData"] = ts.ToEstimatedGridConvergenceData("Pressure", normType: NormType.L2_embedded);
    row["P-Convergence"] = ((Plot2Ddata)row["P-PlotData"]).Regression().ToDictionary(kvp => Convert.ToInt32(kvp.Key), kvp => kvp.Value);

    row["U-PlotData"] = ts.ToEstimatedGridConvergenceData("VelocityX", normType: NormType.L2_embedded);
    row["U-Convergence"] = ((Plot2Ddata)row["U-PlotData"]).Regression().ToDictionary(kvp => Convert.ToInt32(kvp.Key), kvp => kvp.Value);

    row["V-PlotData"] = ts.ToEstimatedGridConvergenceData("VelocityY", normType: NormType.L2_embedded);
    row["V-Convergence"] = ((Plot2Ddata)row["V-PlotData"]).Regression().ToDictionary(kvp => Convert.ToInt32(kvp.Key), kvp => kvp.Value);

    row["T-PlotData"] = ts.ToEstimatedGridConvergenceData("Temperature", normType: NormType.L2_embedded);
    row["T-Convergence"] = ((Plot2Ddata)row["T-PlotData"]).Regression().ToDictionary(kvp => Convert.ToInt32(kvp.Key), kvp => kvp.Value);

    Console.WriteLine("Done with: " + row["Study"]);
}

In [ ]:
foreach(DataRow row in table.Rows){
    var ux = (Plot2Ddata)row["U-PlotData"] ;
    var uy = (Plot2Ddata)row["V-PlotData"] ;
    
    Dictionary<string, double[][]> dataGroups = new Dictionary<string, double[][]>();    
    foreach(var groupX in ux.dataGroups){
        var groupY = uy.dataGroups.Where(g => g.Name == groupX.Name).Single();
        double[] xValues = groupX.Abscissas;
        double[] yValues = groupX.Values.Zip(groupY.Values, (a, b) => Math.Sqrt(a*a+b*b)).ToArray();
        dataGroups.Add(groupX.Name, new double[2][] { xValues, yValues });
    }
    
    row["Velocity-PlotData"] = new Plot2Ddata(dataGroups.ToArray()).WithLogX().WithLogY();
    row["Velocity-Convergence"] = ((Plot2Ddata)row["Velocity-PlotData"]).Regression().ToDictionary(kvp => Convert.ToInt32(kvp.Key), kvp => kvp.Value);

}

In [ ]:
static IEnumerable<DataRow> ExtractSessionsForStage(this DataTable table, int stage){

    switch(stage){
        case 0:{
        return table.AsEnumerable().Where(row => row.Field<string>("Study").Contains("Case0"));
        }
        case 1:{
        return table.AsEnumerable().Where(row => row.Field<string>("Study").Contains("Case1"));
        }
        case 2:{
        return table.AsEnumerable().Where(row => row.Field<string>("Study").Contains("Case2"));
        }
        default:{
            throw new NotImplementedException();
        }
    }
}

static void VizualizeStage(this DataTable table, int stage){
    Console.WriteLine("".PadRight(100,'/'));
    Console.WriteLine("".PadRight(100,'\\'));
    Console.WriteLine("Results for stage {0}", stage);
    
    var rows = table.ExtractSessionsForStage(stage);
    
    foreach(DataRow row in rows){
        Console.WriteLine("".PadRight(100,'/'));
        Console.WriteLine("".PadRight(100,'\\'));
        Console.WriteLine(row["Study"]);

        foreach(var kvp in ((Dictionary<int, double>)row["P-Convergence"])){
            Console.WriteLine($"{kvp.Key} : {kvp.Value}");
        }
        ((Plot2Ddata)row["P-PlotData"]).ModFormat();
        ((Plot2Ddata)row["P-PlotData"]).PlotNow().Display();

        foreach(var kvp in ((Dictionary<int, double>)row["Velocity-Convergence"])){
            Console.WriteLine($"{kvp.Key} : {kvp.Value}");
        }
        ((Plot2Ddata)row["Velocity-PlotData"]).ModFormat();
        ((Plot2Ddata)row["Velocity-PlotData"]).PlotNow().Display();

        foreach(var kvp in ((Dictionary<int, double>)row["T-Convergence"])){
            Console.WriteLine($"{kvp.Key} : {kvp.Value}");
        } 
        ((Plot2Ddata)row["T-PlotData"]).ModFormat();
        ((Plot2Ddata)row["T-PlotData"]).PlotNow().Display();
    }
}

static void WriteTexTableForStage(this DataTable table){

    var rows0 = table.ExtractSessionsForStage(0);
    var rows1 = table.ExtractSessionsForStage(1);
    var rows2 = table.ExtractSessionsForStage(2);

    int[] kuS = new int[] {2, 3, 4}; // control which datalines to plot!
    int[] kToffS = new int[] {-1, 0, 1, 2};
    
    Dictionary<int, double>[,,] ConvergenceRatesP = new Dictionary<int, double>[3,kuS.Length,kToffS.Length];
    Dictionary<int, double>[,,] ConvergenceRatesU = new Dictionary<int, double>[3,kuS.Length,kToffS.Length];
    Dictionary<int, double>[,,] ConvergenceRatesT = new Dictionary<int, double>[3,kuS.Length,kToffS.Length];

    for(int i = 0; i < kuS.Length; i++){
        for(int j = 0; j < kToffS.Length; j++){
            var row = rows0.Where(r => ((string)r["Study"]).Contains("ku"+kuS[i]) && ((string)r["Study"]).Contains("kT"+(kuS[i]+kToffS[j]))).Single();
            ConvergenceRatesP[0,i,j] = (Dictionary<int, double>)row["P-Convergence"];
            ConvergenceRatesU[0,i,j] = (Dictionary<int, double>)row["Velocity-Convergence"];
            ConvergenceRatesT[0,i,j] = (Dictionary<int, double>)row["T-Convergence"];
            
            row = rows1.Where(r => ((string)r["Study"]).Contains("ku"+kuS[i]) && ((string)r["Study"]).Contains("kT"+(kuS[i]+kToffS[j]))).Single();
            ConvergenceRatesP[1,i,j] = (Dictionary<int, double>)row["P-Convergence"];
            ConvergenceRatesU[1,i,j] = (Dictionary<int, double>)row["Velocity-Convergence"];
            ConvergenceRatesT[1,i,j] = (Dictionary<int, double>)row["T-Convergence"];
            
            row = rows2.Where(r => ((string)r["Study"]).Contains("ku"+kuS[i]) && ((string)r["Study"]).Contains("kT"+(kuS[i]+kToffS[j]))).Single();
            ConvergenceRatesP[2,i,j] = (Dictionary<int, double>)row["P-Convergence"];
            ConvergenceRatesU[2,i,j] = (Dictionary<int, double>)row["Velocity-Convergence"];
            ConvergenceRatesT[2,i,j] = (Dictionary<int, double>)row["T-Convergence"];
        }
    }


    string WriteRowP(int setup){
        string rr = "";
        for(int i = 0; i < kuS.Length; i++){
            for(int j = 0; j < kToffS.Length; j++){
                rr += $@"& {ConvergenceRatesP[setup,i,j][kuS[i]-1].ToString("N2")}"; // should only contain one value anyway
            }            
        }
        rr += @"\\";
        return rr;
    }

    string WriteRowU(int setup){
        string rr = "";
        for(int i = 0; i < kuS.Length; i++){
            for(int j = 0; j < kToffS.Length; j++){
                rr += $@"& {ConvergenceRatesU[setup,i,j][kuS[i]].ToString("N2")}"; // should only contain one value anyway
            }            
        }
        rr += @"\\";
        return rr;
    }
    
    string WriteRowT(int setup){
        string rr = "";
        for(int i = 0; i < kuS.Length; i++){
            for(int j = 0; j < kToffS.Length; j++){
                rr += $@"& {ConvergenceRatesT[setup,i,j][kuS[i]+kToffS[j]].ToString("N2")}"; // should only contain one value anyway
            }            
        }
        rr += @"\\";
        return rr;
    }

    string WriteHeader(){
        string rr = "";
        for(int i = 0; i < kuS.Length; i++){
            rr += @" & \multicolumn{"+kToffS.Length+@"}{ c }{$" + kuS[i] + @" \mid "+ (kuS[i]-1) +@"$}";
        }
        rr += @"\\";
        return rr;
    }

    string WriteHeaderT(){
        string rr = "";
        for(int i = 0; i < kuS.Length; i++){
            for(int j = 0; j < kToffS.Length; j++){
                rr += $@" & ${kuS[i]+kToffS[j]}$";
            }
        }
        rr += @"\\";
        return rr;
    }
    
    using(var stw = new StringWriter()) {    
    
        stw.WriteLine(@"\begin{table}[t!]");
        stw.WriteLine(@"    \centering");
        stw.WriteLine(@"    \caption{EOC, for the temperature velocity coupling.}");
        stw.WriteLine(@"    \label{tab:singlephasebc:2}");
        stw.WriteLine(@"    \begin{tabularx}{\linewidth}{ >{\hsize=2.5\hsize \raggedleft\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X  >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X >{\hsize=.875\hsize\centering\arraybackslash}X }		");
        stw.WriteLine(@"        $\Pdeg_{\veloc} \mid \Pdeg_{\press}$ " + WriteHeader());
        //stw.WriteLine(@"        \cmidrule(lr){2-4}\cmidrule(lr){5-7}\cmidrule(lr){8-10}\cmidrule(lr){11-13}"); // adjust manually!
        stw.WriteLine(@"        \cmidrule(lr){2-5}\cmidrule(lr){6-9}\cmidrule(lr){10-13}"); // adjust manually!
        stw.WriteLine(@"        $\Pdeg_{\temp}$" + WriteHeaderT());
        stw.WriteLine(@"        \rowcolor{gray!20} & & & & & & & & & & & &\\[-1em]");
        stw.WriteLine(@"        \rowcolor{gray!20} \multicolumn{1}{ l }{$\|\press-\press_{ref}\|$} & & & & & & & & & & & & \\");
        for(int i = 0; i < 3; i++){
            stw.WriteLine(@"        \rowcolor{gray!20}"+$@" $T_{i}$" + WriteRowP(i));
        }
        stw.WriteLine(@"        & & & & & & & & & & & &\\[-1em]");
        stw.WriteLine(@"        \multicolumn{1}{ l }{$\|\veloc-\veloc_{ref}\|$} & & & & & & & & & & & & \\");
        for(int i = 0; i < 3; i++){
            stw.WriteLine(@"        "+$@" $T_{i}$" + WriteRowU(i));
        }
        stw.WriteLine(@"        \rowcolor{gray!20} & & & & & & & & & & & & \\[-1em]");
        stw.WriteLine(@"        \rowcolor{gray!20} \multicolumn{1}{ l }{$\|\temp-\temp_{ref}\|$} & & & & & & & & & & & &\\");
        for(int i = 0; i < 3; i++){
            stw.WriteLine(@"        \rowcolor{gray!20}"+$@" $T_{i}$" + WriteRowT(i));
        }
        stw.WriteLine(@"    \end{tabularx}");
        stw.WriteLine(@"\end{table}	");
        
        File.WriteAllText($"{BoSSSshell.wmg.CurrentProject}Table.tex", stw.ToString());
    }
}

Export tex table

In [ ]:
//WriteTexTableForStage(table);

In [ ]:
var rows1 = table.ExtractSessionsForStage(1);
var row = rows1.Where(r => ((string)r["Study"]).Contains("ku"+3) && ((string)r["Study"]).Contains("kT"+3)).Single();
var ts = ((List<ITimestepInfo>)row["Timesteps"]).Where(t => t.Grid.NumberOfCells == 32*32).Single();

In [ ]:
//ts.Export().To(Path.GetFullPath(Path.Join(".",ts.Session.Name))).WithSupersampling(2).Do();

In [ ]:
//ts.Export().To(Path.GetFullPath(Path.Join(".",ts.Session.Name))).WithSupersampling(0).Do();


Assert slopes to state of thesis $\pm0.1$  
Field order ist $(p, u, T)$

In [ ]:
Dictionary<string, double[]> ControlSlopes = new Dictionary<string, double[]>();
ControlSlopes["ku2_kT1_Case0"] = new double[]{-0.02, 0.76, 1.41};
ControlSlopes["ku2_kT2_Case0"] = new double[]{1.06, 2.1, 3.43};
ControlSlopes["ku2_kT3_Case0"] = new double[]{1.87, 3.21, 3.88};
ControlSlopes["ku2_kT4_Case0"] = new double[]{2.09, 3.4, 4.96};
ControlSlopes["ku2_kT1_Case1"] = new double[]{0.14, 0.99, 1.44};
ControlSlopes["ku2_kT2_Case1"] = new double[]{0.92, 2.09, 3.36};
ControlSlopes["ku2_kT3_Case1"] = new double[]{1.78, 3.13, 3.96};
ControlSlopes["ku2_kT4_Case1"] = new double[]{2.47, 2.9, 4.91};
ControlSlopes["ku2_kT1_Case2"] = new double[]{0.2, 0.93, 1.18};
ControlSlopes["ku2_kT2_Case2"] = new double[]{0.95, 2.08, 3.15};
ControlSlopes["ku2_kT3_Case2"] = new double[]{1.77, 3.14, 3.35};
ControlSlopes["ku2_kT4_Case2"] = new double[]{2.48, 2.85, 3.12};


ControlSlopes["ku3_kT2_Case0"] = new double[]{1.08, 2.1, 3.43};
ControlSlopes["ku3_kT3_Case0"] = new double[]{1.85, 2.88, 3.88};
ControlSlopes["ku3_kT4_Case0"] = new double[]{2.98, 4.03, 4.96};
ControlSlopes["ku3_kT5_Case0"] = new double[]{2.99, 4.06, 5.76};
ControlSlopes["ku3_kT2_Case1"] = new double[]{0.81, 2.12, 3.36};
ControlSlopes["ku3_kT3_Case1"] = new double[]{1.64, 3.2, 3.96};
ControlSlopes["ku3_kT4_Case1"] = new double[]{2.94, 3.98, 4.91};
ControlSlopes["ku3_kT5_Case1"] = new double[]{2.85, 3.71, 5.79};
ControlSlopes["ku3_kT2_Case2"] = new double[]{0.85, 2.11, 3.15};
ControlSlopes["ku3_kT3_Case2"] = new double[]{1.59, 3.21, 3.35};
ControlSlopes["ku3_kT4_Case2"] = new double[]{2.88, 3.92, 3.12};
ControlSlopes["ku3_kT5_Case2"] = new double[]{2.72, 3.59, 3.05};

ControlSlopes["ku4_kT3_Case0"] = new double[]{1.53, 2.85, 3.88};
ControlSlopes["ku4_kT4_Case0"] = new double[]{3.06, 4.12, 4.96};
ControlSlopes["ku4_kT5_Case0"] = new double[]{3.4, 4.88, 5.76};
ControlSlopes["ku4_kT6_Case0"] = new double[]{2.46, 3.63, 5.39};
ControlSlopes["ku4_kT3_Case1"] = new double[]{1.32, 3.22, 3.96};
ControlSlopes["ku4_kT4_Case1"] = new double[]{2.89, 4.21, 4.91};
ControlSlopes["ku4_kT5_Case1"] = new double[]{3.22, 4.97, 5.79};
ControlSlopes["ku4_kT6_Case1"] = new double[]{2.44, 3.93, 6.29};
ControlSlopes["ku4_kT3_Case2"] = new double[]{1.27, 3.24, 3.35};
ControlSlopes["ku4_kT4_Case2"] = new double[]{2.9, 4.17, 3.12};
ControlSlopes["ku4_kT5_Case2"] = new double[]{3.05, 4.81, 3.05};
ControlSlopes["ku4_kT6_Case2"] = new double[]{2.42, 3.97, 3.01};

In [ ]:
// todo load row from table and compare slopes
bool success = true;
double thrsh = 0.1;

foreach(var kvp in ControlSlopes){
    double[] cVals = kvp.Value;
    var row = table.AsEnumerable().Where(row => row.Field<string>("Study").Equals(kvp.Key)).Single();
    double[] eVals = new double[]{((Dictionary<int, double>)row["P-Convergence"]).First().Value, ((Dictionary<int, double>)row["Velocity-Convergence"]).First().Value, ((Dictionary<int, double>)row["T-Convergence"]).First().Value};

    for(int i = 0; i < eVals.Length; i++){
        if(eVals[i] < cVals[i] - thrsh || eVals[i] > cVals[i] + thrsh){
            success &= false;
            Console.WriteLine("Mismatch in computed slopes in {0}, for field {1}", kvp.Key, i);
        }
    }
}

if(!success){
    throw new ApplicationException("Not all slopes match!");
}
